# Álgebra en FEC (II): Guía de Ejercicios resuelta con `fec_algebra`

Este notebook resuelve la Guía de Ejercicios de Álgebra en FEC usando la librería
`fec_algebra` armada en el TP1 (`GF`, `GFElement`, `GFPoly`).

Los ejercicios 1 a 6 usan la librería tal cual. Los ejercicios 7 y 8 (espacios
vectoriales y códigos lineales) se salen un poco del alcance de la librería del
TP1, que no maneja vectores ni matrices, así que ahí agrego unas funciones
auxiliares chicas armadas sobre `GFElement` para la aritmética escalar (suma =
XOR, producto = AND en GF(2)), sin tocar el paquete en sí.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), ".."))
sys.path.insert(0, "..")  # para poder importar fec_algebra parado en notebooks/

from fec_algebra import GF, GFElement, GFPoly

## Ejercicio 1: Polinomios sobre GF(2)

Cada secuencia de bits $b_0 b_1 \cdots b_n$ se representa como
$b(X) = b_0 + b_1 X + \cdots + b_n X^n$ (el bit más a la izquierda es el
coeficiente de $X^0$).

Uso `GF(m=1, primitive_poly=0)` para representar $GF(2)$: con $m=1$ los
elementos son bits sueltos y la reducción nunca se activa (el producto de
dos bits nunca pasa de grado 0), así que el polinomio primitivo no importa
y queda en 0.

In [ ]:
GF2 = GF(m=1, primitive_poly=0)

def poly_from_bitstring(bits, field=GF2):
    """bits[i] es el coeficiente de X^i (bit más a la izquierda = X^0).
    GFPoly espera los coeficientes en orden DECRECIENTE [a_n,...,a0],
    así que invertimos la secuencia recibida."""
    coeffs_ascendente = [int(c) for c in bits]
    return GFPoly(field, list(reversed(coeffs_ascendente)))

def poly_to_bitstring(p, length):
    """Inversa de poly_from_bitstring: p[i] es el coeficiente de X^i."""
    return "".join(str(int(p[i])) for i in range(length))

def poly_str(p):
    """Representación legible '1 + X + X^3 + ...' para depurar a ojo."""
    terms = []
    for i in range(p.degree, -1, -1):
        if int(p[i]) == 1:
            terms.append("1" if i == 0 else ("X" if i == 1 else f"X^{i}"))
    return " + ".join(terms) if terms else "0"

**a)** Obtener $a(X)$ y $b(X)$.

In [ ]:
a_bits = "1101010"
b_bits = "01101101"

aX = poly_from_bitstring(a_bits)
bX = poly_from_bitstring(b_bits)

print("a(X) =", poly_str(aX))
print("b(X) =", poly_str(bX))

**b)** Calcular $a(X) + b(X)$ y expresarlo como secuencia de bits.

En $GF(2)$ sumar polinomios es hacer XOR coeficiente a coeficiente, que es
justo lo que hace `GFPoly.__add__`.

In [ ]:
suma = aX + bX
print("a(X) + b(X) =", poly_str(suma))
print("como bits (longitud 8):", poly_to_bitstring(suma, length=8))

**c)** Calcular el producto $a(X) \cdot b(X)$.

In [ ]:
producto = aX * bX
print("a(X) * b(X) =", poly_str(producto))
print("grado:", producto.degree)

**d)** Dividir $a(X)\cdot b(X)$ por $a(X)$ y verificar que se recupera $b(X)$
con resto cero.

In [ ]:
q, r = divmod(producto, aX)
print("cociente q(X) =", poly_str(q))
print("resto    r(X) =", poly_str(r))
print("¿q(X) == b(X)? ", q == bX)
print("¿resto nulo?   ", r.degree == -1)

## Ejercicio 2: División de polinomios

$f(X) = 1 + X^2 + X^3 + X^5 + X^6$, $\; g(X) = 1 + X + X^3$.

In [ ]:
def poly_from_terms(terms, field=GF2):
    """terms: dict {grado: coeficiente}. Arma el GFPoly a partir de los
    términos no nulos, sin tener que escribir la lista completa a mano."""
    grado_max = max(terms.keys())
    coeffs_desc = [terms.get(g, 0) for g in range(grado_max, -1, -1)]
    return GFPoly(field, coeffs_desc)

f = poly_from_terms({0: 1, 2: 1, 3: 1, 5: 1, 6: 1})
g = poly_from_terms({0: 1, 1: 1, 3: 1})

print("f(X) =", poly_str(f))
print("g(X) =", poly_str(g))

**a)** Calcular cociente y resto.

In [ ]:
q, r = divmod(f, g)
print("q(X) =", poly_str(q))
print("r(X) =", poly_str(r))

**b)** Verificar la identidad $f(X) = q(X)\,g(X) + r(X)$.

In [ ]:
print("¿f == q*g + r ?", (q * g + r) == f)

## Ejercicio 3: Polinomios irreducibles y primitivos

$p(X) = 1 + X^3 + X^4$, usado como primitivo para construir $GF(2^4)$.

Según Lin & Costello (*Error Control Coding*, cap. 2, "Introduction to
Algebra"), un polinomio primitivo es el de grado mínimo del que un elemento
primitivo $\alpha$ (generador del grupo multiplicativo de $GF(2^m)$) es raíz.
No alcanza con que $p(X)$ sea irreducible nada más: si no es también
primitivo, $\alpha$ no llega a generar los $2^m-1$ elementos no nulos, y ahí
se rompe sin avisar cualquier cuenta basada en tabla de logaritmos. Por eso
en b) y c) primero verifico irreducibilidad, y en c) chequeo además que el
orden de $\alpha$ dé $2^4-1=15$.

**a)** Verificar que $p(X)$ no tiene raíces en $GF(2)$: alcanza con evaluar
en 0 y en 1.

In [ ]:
p = poly_from_terms({0: 1, 3: 1, 4: 1})
print("p(X) =", poly_str(p))
print("p(0) =", int(p(GF2(0))))
print("p(1) =", int(p(GF2(1))))
print("¿tiene raíces en GF(2)? ", int(p(GF2(0))) == 0 or int(p(GF2(1))) == 0)

**b)** Verificar que $p(X)$ no es divisible por el único irreducible de
grado 2, $X^2+X+1$ (si lo fuera, el resto de la división daría 0).

In [ ]:
irreducible_g2 = poly_from_terms({0: 1, 1: 1, 2: 1})  # X^2 + X + 1
_, resto = divmod(p, irreducible_g2)
print("resto de p(X) / (X^2+X+1) =", poly_str(resto))
print("¿divisible?", resto.degree == -1)
print("=> p(X) es irreducible (no tiene raíces y no es divisible por el único")
print("   irreducible de grado 2, la única factorización posible de grado 4).")

**c)** Tabla de las tres representaciones de $GF(2^4)$: potencia de
$\alpha$, representación entera y vector binario.

Lin & Costello (cap. 2) describen dos formas de representar un elemento del
campo: la vectorial (los $m$ bits/coeficientes, la que usa `GFElement`
internamente, cómoda para sumar por XOR) y la exponencial (como potencia de
$\alpha$, cómoda para multiplicar y dividir sumando y restando exponentes).
Esta tabla es el puente entre las dos: se arma potencia por potencia,
multiplicando por $\alpha$ y reduciendo con el primitivo cuando se pasa del
grado $m-1$.

Armo $GF(2^4)$ con `primitive_poly` igual a los bits de $p(X)$ sin el
término $X^4$ implícito: $1+X^3 \Rightarrow$ bits $(b_3,b_2,b_1,b_0) =
(1,0,0,1) = \texttt{0b1001}$. $\alpha$ es la raíz del primitivo, representada
por el elemento $X$, o sea el entero 2 (bits `0010`).

In [ ]:
GF16 = GF(m=4, primitive_poly=0b1001)
alpha = GF16(2)

print(f"{'potencia':<10}{'entero':<8}{'binario':<10}")
elem = GF16(1)
for i in range(15):
    print(f"alpha^{i:<4}{int(elem):<8}{int(elem):04b}")
    elem = elem * alpha

# chequeo extra: alpha tiene que tener orden 15 (generador del grupo
# multiplicativo), coherente con que p(X) sea primitivo y no solo irreducible
orden = None
cur = GF16(1)
for i in range(1, 16):
    cur = cur * alpha
    if int(cur) == 1:
        orden = i
        break
print("\norden de alpha:", orden, "(debe ser 15)")

## Ejercicio 4: Suma y producto de polinomios sobre $GF(2^4)$

$f(X) = \alpha^3 X^2 + \alpha^7 X + \alpha^2$, $\;g(X) = \alpha^5 X + \alpha^{10}$.

Para expresar los coeficientes como potencias de $\alpha$ armo una tabla de
logaritmo discreto (inversa de la tabla de potencias del ejercicio 3).

In [ ]:
log_table = {}
elem = GF16(1)
for i in range(15):
    log_table[int(elem)] = i
    elem = elem * alpha

def alpha_pow(n):
    """alpha^n, con el exponente tomado mod 15 (orden del grupo multiplicativo)."""
    return alpha ** (n % 15)

def as_alpha_str(x):
    """Representa un GFElement no nulo de GF16 como 'alpha^i' (o '1' si i=0)."""
    v = int(x)
    if v == 0:
        return "0"
    exp = log_table[v]
    return "1" if exp == 0 else f"alpha^{exp}"

def poly_alpha_str(p):
    """Representa un GFPoly sobre GF16 con coeficientes como potencias de alpha."""
    terms = []
    for i in range(p.degree, -1, -1):
        c = p[i]
        if int(c) == 0:
            continue
        coef = as_alpha_str(c)
        if i == 0:
            terms.append(coef)
        elif i == 1:
            terms.append(f"{coef}*X" if coef != "1" else "X")
        else:
            terms.append(f"{coef}*X^{i}" if coef != "1" else f"X^{i}")
    return " + ".join(terms) if terms else "0"

In [ ]:
f = GFPoly(GF16, [alpha_pow(3), alpha_pow(7), alpha_pow(2)])
g = GFPoly(GF16, [alpha_pow(5), alpha_pow(10)])

print("f(X) =", poly_alpha_str(f))
print("g(X) =", poly_alpha_str(g))

**a)** Calcular $f(X) + g(X)$.

In [ ]:
suma = f + g
print("f(X) + g(X) =", poly_alpha_str(suma))

**b)** Calcular $f(X) \cdot g(X)$.

In [ ]:
producto = f * g
print("f(X) * g(X) =", poly_alpha_str(producto))

## Ejercicio 5: División y potencia de polinomios sobre $GF(2^4)$

**a)** Dividir $f(X) = \alpha^3 X^3 + \alpha^9 X^2 + X + \alpha^6$ por
$g(X) = X + \alpha^5$.

In [ ]:
f5 = GFPoly(GF16, [alpha_pow(3), alpha_pow(9), GF16(1), alpha_pow(6)])
g5 = GFPoly(GF16, [GF16(1), alpha_pow(5)])

q5, r5 = divmod(f5, g5)
print("f(X) =", poly_alpha_str(f5))
print("g(X) =", poly_alpha_str(g5))
print("q(X) =", poly_alpha_str(q5))
print("r(X) =", poly_alpha_str(r5))
print("¿f == q*g + r ?", (q5 * g5 + r5) == f5)

**b)** Calcular $h(X)^2$ para $h(X) = \alpha^3 X^2 + \alpha^{11} X + \alpha^6$,
multiplicando $h(X)$ por sí mismo.

In [ ]:
h = GFPoly(GF16, [alpha_pow(3), alpha_pow(11), alpha_pow(6)])
h2 = h * h
print("h(X)   =", poly_alpha_str(h))
print("h(X)^2 =", poly_alpha_str(h2))

**c)** Verificar la identidad de Frobenius $(h(X))^2 = h(X^2)$: elevar al
cuadrado cada coeficiente por separado y ubicarlo en el doble del grado
correspondiente.

In [ ]:
terms_h2 = {}
for i in range(h.degree, -1, -1):
    c = h[i]
    if int(c) != 0:
        terms_h2[2 * i] = c * c   # coeficiente al cuadrado, en grado 2*i

grado_max = max(terms_h2.keys())
coeffs = [terms_h2.get(gi, GF16(0)) for gi in range(grado_max, -1, -1)]
h_de_x2 = GFPoly(GF16, coeffs)

print("h(X^2) armado término a término =", poly_alpha_str(h_de_x2))
print("¿coincide con h(X)^2 calculado por producto?", h_de_x2 == h2)

## Ejercicio 6: Búsqueda de raíces

Encontrar las raíces de $f(X) = X^2 + \alpha^5 X + \alpha^9$.

Como $GF(2^4)$ tiene solo 16 elementos, alcanza con evaluar el polinomio en
cada uno de ellos y quedarme con los que dan 0. Es fuerza bruta, pero para
un campo tan chico no hace falta nada más elaborado.

In [ ]:
f6 = GFPoly(GF16, [GF16(1), alpha_pow(5), alpha_pow(9)])
print("f(X) =", poly_alpha_str(f6))

raices = [GF16(v) for v in range(GF16.order) if f6(GF16(v)) == GF16(0)]
for r in raices:
    print("raíz:", as_alpha_str(r), " (entero", int(r), ")")

# chequeo extra: para un cuadrático mónico X^2+bX+c, la suma de las raíces
# tiene que dar b (en GF(2) restar es sumar)
print("\nsuma de las raíces =", as_alpha_str(raices[0] + raices[1]), " (debe ser alpha^5)")

## Ejercicio 7: Ortogonalidad sobre GF(2)

Este ejercicio trabaja con vectores, no con polinomios, así que no uso
`GFPoly`. Sí reutilizo `GFElement` sobre `GF2` para que cada componente sea
un elemento de campo real (con su suma y producto ya probados), en vez de
manipular ints a mano.

**a)** Producto interno de $u=(1,1,0,1)$ y $v=(0,1,1,1)$ en $V_4$ sobre $GF(2)$.

In [ ]:
def inner_product(u, v, field=GF2):
    """Producto interno sobre el campo: suma (XOR) de los productos
    componente a componente."""
    acc = field(0)
    for a, b in zip(u, v):
        acc = acc + a * b
    return acc

u = [GF2(1), GF2(1), GF2(0), GF2(1)]
v = [GF2(0), GF2(1), GF2(1), GF2(1)]

print("<u,v> =", int(inner_product(u, v)))

**b)** Vector no nulo de $V_2$ ortogonal a sí mismo.

Sobre los reales, $\langle x,x\rangle = \sum x_i^2 \geq 0$, y da 0 únicamente
si $x=0$ (suma de cuadrados no negativos). En $GF(2)$ eso se rompe: como
$1+1=0$, un vector con un número par de unos puede anularse consigo mismo
sin ser el vector nulo.

In [ ]:
candidato = [GF2(1), GF2(1)]
print("x =", [int(c) for c in candidato])
print("<x,x> =", int(inner_product(candidato, candidato)))
print("¿x es no nulo?", any(int(c) != 0 for c in candidato))

## Ejercicio 8: Matriz generadora, forma sistemática y matriz de verificación

Según Lin & Costello (cap. 3, "Linear Block Codes"), un código lineal $(n,k)$
es un subespacio de $GF(2)^n$ definido por una matriz generadora $G$
($k\times n$, cuyo espacio de filas es el código) y una matriz de
verificación $H$ ($(n-k)\times n$, que expande el complemento ortogonal, el
código dual), relacionadas por $GH^T=0$. Llevando $G$ a forma sistemática
$[I_k \mid P]$, se puede mostrar que $H=[P^T \mid I_{n-k}]$ cumple esa
relación directamente, que es justo lo que verifico en el punto c).

Trabajar con matrices se va del alcance de la librería del TP1 (no tiene una
clase de matrices), así que para no meterme con el paquete armé estas
funciones auxiliares acá en el notebook. Usan `GFElement` sobre `GF2` como
escalares: la suma y el producto siguen siendo los de la librería, solo que
organizados en filas y columnas.

In [ ]:
def to_field_matrix(rows, field=GF2):
    return [[field(x) for x in row] for row in rows]

def matrix_str(M):
    return "\n".join(" ".join(str(int(x)) for x in row) for row in M)

def transpose(M):
    return [list(row) for row in zip(*M)]

def mat_mult(A, B, field=GF2):
    """Producto de matrices A (r x n) por B (n x c) sobre el campo dado."""
    r, n, c = len(A), len(A[0]), len(B[0])
    result = [[field(0)] * c for _ in range(r)]
    for i in range(r):
        for j in range(c):
            acc = field(0)
            for k in range(n):
                acc = acc + A[i][k] * B[k][j]
            result[i][j] = acc
    return result

def vec_mat_mult(v, M, field=GF2):
    return mat_mult([v], M, field)[0]

def row_reduce_to_systematic(M, k):
    """Lleva las primeras k columnas de M a la identidad I_k mediante
    operaciones elementales de fila (sumar una fila a otra)."""
    M = [row[:] for row in M]
    for col in range(k):
        pivot_row = next((r for r in range(col, len(M)) if int(M[r][col]) == 1), None)
        if pivot_row is None:
            raise ValueError(f"No hay pivote para la columna {col}: la matriz no tiene rango completo.")
        M[col], M[pivot_row] = M[pivot_row], M[col]
        for r in range(len(M)):
            if r != col and int(M[r][col]) == 1:
                M[r] = [a + b for a, b in zip(M[r], M[col])]
    return M

**a)** Llevar $G$ a la forma sistemática $[I_3 \mid P]$ (Lin & Costello, cap.
3: toda matriz generadora de rango completo puede llevarse a esta forma con
operaciones elementales de fila, sin perder generalidad, porque el código
resultante es equivalente al original).

In [ ]:
G = to_field_matrix([
    [1, 1, 0, 0, 1, 1],
    [0, 1, 1, 1, 0, 1],
    [0, 0, 1, 0, 1, 1],
])
print("G =")
print(matrix_str(G))

Gsist = row_reduce_to_systematic(G, k=3)
print("\nGsist = [I3 | P] =")
print(matrix_str(Gsist))

**b)** A partir de $P$, construir $H = [P^T \mid I_3]$ (Cap. 3: esta
construcción de $H$ a partir de la forma sistemática de $G$ es la que
garantiza $GH^T=0$ sin necesidad de resolverlo por otro método).

In [ ]:
P = [row[3:] for row in Gsist]
PT = transpose(P)
I3 = to_field_matrix([[1,0,0],[0,1,0],[0,0,1]])
H = [PT[i] + I3[i] for i in range(3)]

print("P =")
print(matrix_str(P))
print("\nH = [P^T | I3] =")
print(matrix_str(H))

**c)** Verificar que $G_{sist} \cdot H^T = 0$, mostrando explícitamente al
menos dos de los nueve productos internos involucrados (cada entrada de
$G_{sist}\cdot H^T$ es, por definición, uno de esos productos internos: fila
de $G_{sist}$ contra fila de $H$).

In [ ]:
HT = transpose(H)

# Dos productos internos explícitos: fila 0 de Gsist con fila 0 de H,
# y fila 1 de Gsist con fila 2 de H.
p_00 = inner_product(Gsist[0], H[0], GF2)
p_12 = inner_product(Gsist[1], H[2], GF2)
print("fila0(Gsist) . fila0(H) =", int(p_00))
print("fila1(Gsist) . fila2(H) =", int(p_12))

prod_completo = mat_mult(Gsist, HT, GF2)
print("\nGsist . H^T (matriz completa) =")
print(matrix_str(prod_completo))

**d)** Codificar $u=(1,0,1)$ como $v = u\cdot G_{sist}$ y verificar
$v\cdot H^T = 0$. Esto es la operación de codificación del Cap. 3
($c = mG$) seguida del cómputo de síndrome ($s = rH^T$): un síndrome nulo
es consistente con que $v$ sea una palabra código válida (condición
necesaria, aunque el libro aclara que no es suficiente para garantizar
ausencia de error en un canal ruidoso).

In [ ]:
u = [GF2(1), GF2(0), GF2(1)]
v = vec_mat_mult(u, Gsist)
print("u =", [int(x) for x in u])
print("v = u . Gsist =", [int(x) for x in v])

sindrome = vec_mat_mult(v, HT)
print("v . H^T =", [int(x) for x in sindrome], " (debe ser el vector nulo)")

**e)** Linealidad de la codificación: $u_1=(1,1,0)\to c_1$,
$u_2=(1,0,0)\to c_2$, $u_3=u_1+u_2\to c_3$; comprobar $c_1+c_2=c_3$.

In [ ]:
u1 = [GF2(1), GF2(1), GF2(0)]
u2 = [GF2(1), GF2(0), GF2(0)]
u3 = [a + b for a, b in zip(u1, u2)]

c1 = vec_mat_mult(u1, Gsist)
c2 = vec_mat_mult(u2, Gsist)
c3 = vec_mat_mult(u3, Gsist)
c1_mas_c2 = [a + b for a, b in zip(c1, c2)]

print("u1 =", [int(x) for x in u1], "-> c1 =", [int(x) for x in c1])
print("u2 =", [int(x) for x in u2], "-> c2 =", [int(x) for x in c2])
print("u3 = u1+u2 =", [int(x) for x in u3], "-> c3 =", [int(x) for x in c3])
print("c1+c2 =", [int(x) for x in c1_mas_c2])
print("¿c1+c2 == c3?", c1_mas_c2 == c3)